In [ ]:
import os
import pandas as pd

# Cap Polars to 6 threads
os.environ["POLARS_MAX_THREADS"] = "6"

import polars as pl

# loading the data set, using 2024 data
df_lazy = pl.scan_parquet("aisdk-parquet/aisdk-2024-1h.parquet")

# Quick sanity check: pull just the first 100k rows into memory
# to inspect schema and spot any obvious data quality issues
df_sample = df_lazy.head(100_000).collect()

# checking ship types
df_lazy.group_by("Ship type").len().sort("len", descending=True).collect()

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import plotly.graph_objects as go
import plotly.io as pio
import numpy as np

pio.renderers.default = "vscode"

# Haversine computes the great-circle distance between two lat/lon points.
# Input arrays allow vectorised calculation across an entire route at once.
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth's mean radius in km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

# Sums the segment-by-segment distances for a vessel's full route.
# lats[:-1] / lons[:-1] are the "from" points; lats[1:] / lons[1:] are the "to" points.
def total_distance(df):
    lats = df["Latitude"].values
    lons = df["Longitude"].values
    distances = haversine(lats[:-1], lons[:-1], lats[1:], lons[1:])
    return distances.sum()

# Reusable filter applied to every query
# Optionally narrows to a specific MMSI when the user picks a ship.
def base_filter(lf, ship_type, mmsi=None):
    if mmsi:
        lf = lf.filter(pl.col("MMSI") == mmsi)
    return (
        lf
        .filter(pl.col("Ship type") == ship_type)
        .filter(pl.col("Type of mobile") == "Class A") # Class A = large commercial vessels with mandatory, high-quality transponders.
        .filter(
            # Coordinate guards remove corrupt rows (0,0 is a common AIS data artifact).
            pl.col("Latitude").is_between(-90, 90) &
            pl.col("Longitude").is_between(-180, 180) &
            (pl.col("Latitude") != 0) &
            (pl.col("Longitude") != 0)
        )
    )

# Builds the ship dropdown options for a given type.
def build_ship_options(ship_type):
    ship_info = (
        base_filter(df_lazy, ship_type)
        .group_by(["MMSI", "Name"])
        .len()
        .sort("len", descending=True)
        .limit(500) # Caps at 500 vessels to keep the UI responsive.
        .collect()
        .sort("Name")
    )

    options = []
    for row in ship_info.iter_rows(named=True):
        route = (
            base_filter(df_lazy, ship_type, mmsi=row["MMSI"])
            .select(["Latitude", "Longitude"])
            .collect()
            .to_pandas()
        )
        if len(route) > 1: # Silently skips ships with only one ping (can't draw a route)
            dist_km = total_distance(route)
            if dist_km > 10: # and those with total distance < 10 km (stationary/moored vessels).
                options.append((f"{row['Name']} ({row['MMSI']}) — {dist_km:.0f} km", row["MMSI"], dist_km))

    # Final list is sorted by distance descending so the most-travelled ships appear first.
    options.sort(key=lambda x: x[2], reverse=True)
    options = [(label, mmsi) for label, mmsi, _ in options]
    
    return options

# Pull every unique ship type from the dataset for the toggle buttons.
# Filter to Class A only so we don't show types with no plottable vessels.
ship_types = (
    df_lazy
    .filter(pl.col("Type of mobile") == "Class A")
    .select("Ship type")
    .unique()
    .collect()
    .sort("Ship type")
    ["Ship type"]
    .to_list()
)

# Widget definitions
type_selector = widgets.ToggleButtons(
    options=ship_types,
    value="Cargo",           # Default to Cargo as the most data-rich type
    description="Ship type:",
    style={"description_width": "initial", "button_width": "120px"},
)

loading_label = widgets.Label(value="")  # Shows "⏳ Loading..." / "✅ N ships found"

ship_dropdown = widgets.Dropdown(
    description="Ship:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="500px")
)

output = widgets.Output()  # Isolated output cell, prevents plot flickering on update

# Triggered when the user switches ship type.
# Clears and repopulates the dropdown; updates the loading label throughout.
def on_type_change(change):
    loading_label.value = f"⏳ Loading {change['new']} ships..."
    ship_dropdown.options = []
    options = build_ship_options(change["new"])
    ship_dropdown.options = options
    loading_label.value = f"✅ {len(options)} ships found"

# Triggered when the user picks a ship from the dropdown.
# Fetches full route, computes stats, and renders the Plotly map.
def plot_route(change):
    if not change["new"]:
        return
    output.clear_output()
    mmsi = change["new"]
    ship_type = type_selector.value

    route = (
        base_filter(df_lazy, ship_type, mmsi=mmsi)
        .select(["MMSI", "# Timestamp", "Latitude", "Longitude"])
        .sort("# Timestamp")   # Must be chronological for the route line to make sense
        .collect()
        .to_pandas()
    )

    dist_km = total_distance(route)
    duration = route["# Timestamp"].iloc[-1] - route["# Timestamp"].iloc[0]
    hours = duration.total_seconds() / 3600
    avg_speed = dist_km / hours if hours > 0 else 0  # Guard against single-ping edge case

    fig = go.Figure()

    # Main route line
    fig.add_trace(go.Scattermap(
        lat=route["Latitude"],
        lon=route["Longitude"],
        mode="lines",
        line=dict(width=2, color="blue"),
        name="Route",
        hoverinfo="skip",    # Hovering the line itself isn't useful; skip its tooltip
    ))

    # Start marker, green dot with text label
    fig.add_trace(go.Scattermap(
        lat=[route["Latitude"].iloc[0]],
        lon=[route["Longitude"].iloc[0]],
        mode="markers+text",
        marker=dict(size=12, color="green"),
        text=["START"],
        textposition="top right",
        name="Start",
    ))

    # End marker, red dot with text label
    fig.add_trace(go.Scattermap(
        lat=[route["Latitude"].iloc[-1]],
        lon=[route["Longitude"].iloc[-1]],
        mode="markers+text",
        marker=dict(size=12, color="red"),
        text=["END"],
        textposition="top right",
        name="End",
    ))

    fig.update_layout(
        map_style="carto-positron",
        map=dict(
            # Auto-centre the map on the ship's mean position
            center=dict(
                lat=route["Latitude"].mean(),
                lon=route["Longitude"].mean()
            ),
            zoom=5
        ),
        title=f"Route of {ship_dropdown.label} | {dist_km:.0f} km | {duration} | Avg speed: {avg_speed:.1f} km/h",
        margin={"r": 0, "t": 40, "l": 0, "b": 0},
        height=600,
    )

    with output:
        fig.show()

# Wire up the observer callbacks
type_selector.observe(on_type_change, names="value")
ship_dropdown.observe(plot_route, names="value")

# Trigger the initial load manually
loading_label.value = "⏳ Loading Cargo ships..."
ship_dropdown.options = build_ship_options("Cargo")
loading_label.value = f"✅ {len(ship_dropdown.options)} ships found"

display(widgets.VBox([
    type_selector,
    widgets.HBox([ship_dropdown, loading_label]),
    output
]))

# Auto-plot the top ship so the map isn't blank on first open
if ship_dropdown.options:
    plot_route({"new": ship_dropdown.options[0][1]})